In [1]:
#kernel rw_data
import copy
import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.graph_objs import Data
from plotly.subplots import make_subplots

import scipy as sp
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
import scipy.ndimage.filters as spf
import scipy as sp

from scipy.optimize import curve_fit
from sklearn.covariance import MinCovDet
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.tree import plot_tree

import os
import sys
from pathlib import Path

from csv import DictReader, reader
# Define paths
from utils.read_and_create_dict import read_and_create_dict
path_cwd = Path.cwd()
path_input = str(path_cwd) + '/Input_files/'
path_output_graphs = str(path_cwd) + '/Output_graphs/'
path_output_data = str(path_cwd) + '/Output_files/'
path_latex = str(path_cwd.parents[0]) + '/TEX_file/Figures/'

In [2]:
# Import the utility functions
from utils.read_and_create_dict import read_and_create_dict
from utils.clean_psy import clean_psy_in_dict

# Read files, create/update the dictionary, and clean PSY data
trees_data, paths = read_and_create_dict()
trees_data = clean_psy_in_dict(trees_data)

Info: No Soil file specified for DF21-2021 (value: NA)
Info: No Soil file specified for DF03-2021 (value: NA)
Info: No Psy file specified for DF03-2021 (value: NA)
Info: No Soil file specified for ES48-2021 (value: NA)
Info: No Soil file specified for ES42-2021 (value: NA)
Info: No Psy file specified for ES42-2021 (value: NA)
Info: No SF_W file specified for ES51-2022 (value: NA)
Info: No SF_W_incomplete file specified for ES51-2022 (value: NA)
Info: No Soil file specified for ES51-2022 (value: NA)
  PSY data for DF03-2021 is empty or all NaN
  PSY data for ES42-2021 is empty or all NaN
  PSY data for ES51-2022 is None


### Cummulative precipitation and dry wet periods

We identified dry and wet periods within each growing season (2021 and 2022) using cumulative rainfall data from trees_data. Dry periods were defined as spans of at least 10 days with a rainfall increase of ≤ 3 mm, detected by iterating through daily resampled cumulative rainfall and extending windows to find the longest qualifying periods, while remaining periods were classified as wet. The results were visualized in a Plotly scatter plot for each year, displaying the cumulative rainfall (mm) as a blue line over time, with red transparent boxes overlaying dry periods and green transparent boxes marking wet periods, ensuring the entire timeline was covered. The plot’s x-axis showed dates from May to October, and the y-axis ranged up to the maximum cumulative rainfall (e.g., ~250 mm in 2021), providing a clear visual distinction between dry and wet phases for further analysis.

In [3]:
from utils.rainfall import plot_cumulative_rainfall_with_periods

plot_cumulative_rainfall_with_periods(trees_data)

Cumulative Rainfall with Dry Periods for 2021
Cumulative Rainfall with Dry Periods for 2022


({'rainfall_2021': Figure({
      'data': [{'line': {'color': 'blue'},
                'mode': 'lines',
                'name': 'Cumulative Rainfall (mm)',
                'type': 'scatter',
                'x': array([datetime.datetime(2021, 5, 5, 15, 30),
                            datetime.datetime(2021, 5, 5, 16, 0),
                            datetime.datetime(2021, 5, 5, 16, 30), ...,
                            datetime.datetime(2021, 10, 21, 10, 0),
                            datetime.datetime(2021, 10, 21, 10, 30),
                            datetime.datetime(2021, 10, 21, 11, 0)], dtype=object),
                'xaxis': 'x',
                'y': array([  0. ,   0. ,   0. , ..., 249.5, 249.5, 249.5]),
                'yaxis': 'y'}],
      'layout': {'height': 400,
                 'shapes': [{'fillcolor': 'red',
                             'layer': 'below',
                             'line': {'width': 0},
                             'opacity': 0.4,
                    

### Matric potential (Van Genuchten)

Assuming a homogeneous material for the surface fracture, where the sensors were installed, we used the Van Genuchten relationships to calculate the soil matric potential(MPa) from the volumetric water content data (/). 

Data manipulation to consider: 
- to avoid zero division we: 
    1) consider the residual water in soil as the minimum value recorded by the sensor and not the literature pwp 
    2) we added a 0.025 (2% vwc) to this minimum- just above the measurement accuracy of the soil moisture sensors, 
ensuring it's not confounded with actual soil moisture variability. Note that for GS3's Medium Specific Calibration: 
±0.01–0.02 m3/m3 (±1 to 2% VWC) in any porous medium / Generic Calibration: ±0.03 m3/ m3 typical (±3% VWC) typical in 
mineral soils that have solution electrical conductivity < 5 dS/m 
(https://s.campbellsci.com/documents/ca/product-brochures/gs3_br.pdf). 
TEROS12 however has an accuracy of 3% (0.03) (https://metergroup.com/products/teros-12/)

- Taking the residual moisture content to be the minimum value recorded by the sensor can make sense, especially in 
the absence of more detailed soil characterization data, but it's important to compare this with literature values for 
similar soil types to ensure it's within a plausible range.// Keep in mind that the VG parameters give the shape of 
the soil wrc and they are different for each type of soil.

In [4]:
from utils.van_genuchten import calculate_and_store_matric_potential, plot_soil_moisture_and_matric_potential
# Calculate matric potentials and store in trees_data
trees_data = calculate_and_store_matric_potential(trees_data)

# Plot for 2021
vg_fig_2021 = plot_soil_moisture_and_matric_potential(trees_data, '2021')

# Plot for 2022
vg_fig_2022 = plot_soil_moisture_and_matric_potential(trees_data, '2022')

# # save plots to IVC_plots inside Output_graphs
path_VG=path_output_graphs+'IVC_plots/VG/'

for path in [path_VG,path_latex]:
        vg_fig_2021.write_image(path+'VG_matric_potential_2021.png',scale=8)
        vg_fig_2022.write_image(path+'VG_matric_potential_2022.png',scale=8)



### Daily SF WP plots 

In this step, we calculated the predawn water potential (Ψ_pd) for each tree by performing a linear regression on nighttime data (10 PM to 7 AM) using sap flow (SF_incomplete, in cm³/h) as the independent variable (x-axis) and water potential (PSY from PSY_cleaned, in MPa) as the dependent variable (y-axis). The regression equation, PSY = slope * SF_incomplete + intercept, yields the predawn water potential as the intercept (i.e., PSY when SF_incomplete = 0), representing the soil water potential the plant equilibrates with at night. To ensure physical realism, we checked the intercept: if it was positive (which is rare and typically unrealistic for natural conditions as it suggests water pressure rather than tension), we adjusted it to a small negative value of -0.0001 MPa, indicative of wet soil with minimal tension. This adjustment ensures that the predawn water potential aligns with expected physiological conditions for trees, where Ψ_pd is generally negative due to water potential gradients in the soil-plant system. The resulting PSY_predawn values were stored in trees_data for further analysis, such as calculating resistance or constructing vulnerability curves.

We should look at the r2 of each one of these regressions and somehow report it. 

Ψ_pd is determined by intercept, when SF is assumed to be zero. 
but SF is never really zero, Binks et al. say that Ψ_pd= (Ψ_soil-ρgh)+ ΔΨ(FWU)
where ρgh is the change due to gravitational potential (is this really a thing inside the 
vessels that are subject to really high tension?). ΔΨ(FWU) is the water potential change caused by the uptake of
water via foliar water uptake (nighttime transpiration). I will need to correct for these two effects to really 
comparte matric and predawn potentials. 
If I have discarded nighttime transpiration, hydraulic redistribution, and due to a small sapwood (and maybe michelle's data with storage) then we can talk about embolism. 

In [5]:
from utils.predawn import calculate_predawn_water_potential, plot_predawn_scatter

# Calculate predawn water potentials (Ψ_pd) and store in trees_data
trees_data = calculate_predawn_water_potential(trees_data)

# Plot nighttime scatter plots for 2021 and 2022
plots_pd_2021 = plot_predawn_scatter(trees_data, '2021')
plots_pd_2022 = plot_predawn_scatter(trees_data, '2022')

path_predawn = path_output_graphs + 'IVC_plots/Predawn/'
for path in [path_predawn, path_latex]:
    # Save 2021 plots
    for fig, tree_name in plots_pd_2021:
        if fig is not None:  # Ensure the figure is valid
            fig.write_image(path + f'predawn_potential_2021_{tree_name}_2021.png', scale=8)
    
    # Save 2022 plots
    for fig, tree_name in plots_pd_2022:
        if fig is not None:  # Ensure the figure is valid
            fig.write_image(path + f'predawn_potential_2022_{tree_name}_2022.png', scale=8)

#Sap Flow vs. Water Potential (10 PM to 7 AM) for {tree_name} in {year}, Colored by Day


DF49


DF21


DF27


ES48


ES50


...

DF03


ES48


ES50


ES42


ES01


### Plot difference between matric and predawn water potentials

In this step, we plotted the daily average predawn water potential (Ψ_pd) and matric potential (Ψ_matric) for each tree in a given year using data stored in trees_data. The predawn water potential, stored as a pandas Series under trees_data[<tree_year>]['PSY_predawn'], was derived from the intercepts of linear regressions between sap flow (SF_incomplete) and water potential (PSY_cleaned) during nighttime hours (10 PM to 7 AM), with positive intercepts adjusted to -0.0001 MPa to represent wet soil conditions. The matric potential, stored as a DataFrame under trees_data[<tree_year>]['PSY_matric'] with a column 'PSY_matric', represents the soil water potential calculated using the van Genuchten model. For each tree, we computed daily means of both potentials and visualized them on a scatter plot with dates on the x-axis and soil water potential (MPa) on the y-axis, using dots for Ψ_pd (Ψs-intercept) and squares for Ψ_matric (Ψs-VG). The plot revealed significant decoupling between Ψ_pd and Ψ_matric during dry periods, with Ψ_pd often becoming much more negative, likely due to hydraulic disconnection, rooting depth differences, or limitations in the regression and van Genuchten models under drought conditions.

In [6]:
from utils.predawn import plot_psy_matric_daily
# Plot daily averages of PSY_predawn (intercepts) and PSY_matric for 2021 and 2022
both_psy_2021 = plot_psy_matric_daily(trees_data, '2021')
both_psy_2022 = plot_psy_matric_daily(trees_data, '2022')

# # save plots to IVC_plots inside Output_graphs
path_VG=path_output_graphs+'IVC_plots/PD_VG_night/'

for path in [path_VG,path_latex]:
        both_psy_2021.write_image(path+'PD_VG_avg_2021.png',scale=8)
        both_psy_2022.write_image(path+'PD_VG_avg_2022.png',scale=8)



Skipping DF21 (DF21-2021) for 2021 due to no data.
Skipping DF03 (DF03-2021) for 2021 due to no data.
Skipping ES48 (ES48-2021) for 2021 due to no data.
Skipping ES42 (ES42-2021) for 2021 due to no data.


Skipping ES51 (ES51-2022) for 2022 due to no data.


### Disconnection between matric and predawn

In this analysis step, we investigated the hydraulic disconnection between soil and tree water potential by calculating the difference (ΔΨ) between nighttime matric potential (Ψ_soil) and predawn water potential (Ψ_pd) for each tree across the 2021 and 2022 growing seasons. Using the plot_disconnection_drivers function, we aligned hourly measurements of matric potential (derived from soil moisture via the van Genuchten model) and predawn water potential (computed from regression intercepts of sap flow and VPD) over their overlapping time periods, focusing on nighttime hours (10 PM to 7 AM) to minimize transpiration effects. We then visualized ΔΨ against two key environmental drivers—soil moisture and vapor pressure deficit (VPD)—in separate scatter plots, with points colored by the respective driver to highlight their influence. Additionally, we performed linear regression to quantify the relationship between ΔΨ and each driver individually, reporting coefficients, intercepts, and R² values to assess the strength of these relationships. This approach allowed us to explore how soil moisture and atmospheric demand (VPD) contribute to hydraulic disconnection during nighttime, providing insights into the tree's hydraulic status under varying environmental conditions.
/WATER POTENTIAL DIFFERENCE

In [7]:
# from utils.predawn import plot_disconnection_drivers

# # Plot ΔΨ vs. Soil Moisture for 2021 and 2022
# metrics_2021_soil=plot_disconnection_drivers(trees_data, '2021', 'soil')
# metrics_2022_soil=plot_disconnection_drivers(trees_data, '2022', 'soil')

# # Print variability metrics
# # print("\nVariability Metrics (Soil Moisture):")
# # print("2021:", metrics_2021_soil)
# # print("2022:", metrics_2022_soil)


For soil: 
We visualized ΔΨ against soil moisture in scatter plots, with points colored by soil moisture to explore potential trends, and calculated the standard deviation of ΔΨ to assess the consistency of hydraulic disconnection. In 2021, ΔΨ variability ranged from 0.495 MPa (ES51) to 0.937 MPa (ES50), indicating significant fluctuations in hydraulic disconnection for some trees, while in 2022, variability was generally lower, ranging from 0.141 MPa (ES50) to 0.847 MPa (ES42). For instance, ES50 showed a stark contrast between years, with high variability in 2021 (0.937 MPa) but the lowest in 2022 (0.141 MPa), suggesting more stable hydraulic behavior in the latter year. These findings highlight the inconsistent nature of nighttime hydraulic disconnection across trees and years, though missing matric potential data for several trees (e.g., DF21-2021, ES51-2022) limited the analysis for those cases.

In [8]:
# Plot ΔΨ vs. VPD for 2021 and 2022
# from utils.predawn import plot_disconnection_drivers
# plot_disconnection_drivers(trees_data, '2021', 'vpd')
# plot_disconnection_drivers(trees_data, '2022', 'vpd')
# print("\nVariability Metrics (VPD):")
# print("2021:", metrics_2021_vpd)
# print("2022:", metrics_2022_vpd)

A ΔΨ of 1 MPa can have physiological impacts. For example, it may indicate that the tree is not fully rehydrating at night, which could lead to cumulative water stress over time. In species sensitive to embolism, a 1 MPa difference might approach thresholds for cavitation (e.g., if the tree’s 50% loss of conductivity occurs at -2 to -3 MPa, a Ψ_pd of -1.5 MPa is getting close to that threshold).
Compare this ΔΨ to vulnerability curves (which you plan to calculate next). If ΔΨ values are approaching critical thresholds for embolism, the variability is biologically significant.

In [9]:
from utils.predawn import plot_hydraulic_disconnection, compare_hydraulic_disconnection
path_output_graphs = str(path_cwd) + '/Output_graphs/'

# Plot ΔΨ vs. Soil Moisture for 2021 and 2022
plot_metrics_2021 = plot_hydraulic_disconnection(trees_data, '2021')
plot_metrics_2022 = plot_hydraulic_disconnection(trees_data, '2022')
comp_metrics = compare_hydraulic_disconnection(trees_data,path_output=path_output_graphs)




DF27 (2021): Total days with data: 119

Kruskal-Wallis Test for DF27 (2021) - Pre-Dry_vs_Dry: H = 12.249, df = 1, p = 0.00046548 (significant)

Kruskal-Wallis Test for DF27 (2021) - Dry_vs_Post-Dry: H = 9.786, df = 1, p = 0.00175811 (significant)

Kruskal-Wallis Test for DF27 (2021) - Pre-Dry_vs_Post-Dry: H = 1.194, df = 1, p = 0.27448200 (not significant)

ES50 (2021): Total days with data: 107

Kruskal-Wallis Test for ES50 (2021) - Pre-Dry_vs_Dry: H = 3.964, df = 1, p = 0.04648807 (significant)

Kruskal-Wallis Test for ES50 (2021) - Dry_vs_Post-Dry: H = 1.269, df = 1, p = 0.25997128 (not significant)

Kruskal-Wallis Test for ES50 (2021) - Pre-Dry_vs_Post-Dry: H = 8.148, df = 1, p = 0.00431051 (significant)

Metrics for 2021:

DF27 (Cliff):
Average ΔΨ (Pre-Dry): -0.249 MPa
Average ΔΨ (Dry): -0.984 MPa
Average ΔΨ (Post-Dry): -0.287 MPa

ES50 (Cliff):
Average ΔΨ (Pre-Dry): -0.064 MPa
Average ΔΨ (Dry): -0.294 MPa
Average ΔΨ (Post-Dry): -0.218 MPa

DF49 (2022): Total days with data: 111


## IVC

In [10]:
from utils.ivc import calculate_resistance_and_plot


# matric_results = calculate_resistance_and_plot(trees_data, potential_type='matric',sf='complete')
# predawn_results = calculate_resistance_and_plot(trees_data, potential_type='predawn',sf='complete')


#matric_results_incomplete= calculate_resistance_and_plot(trees_data, potential_type='matric',sf='incomplete')
#predawn_results_incomplete= calculate_resistance_and_plot(trees_data, potential_type='predawn',sf='incomplete')

In [11]:
from utils.ivc import calculate_resistance_dry_rain

results_dry_wet=calculate_resistance_dry_rain(trees_data, potential_type='matric', sf='incomplete')

Processing tree: DF49, Year: 2021, Potential Type: matric, SF Type: incomplete
Tree DF49 (Year 2021, Dry Period): PSYx range: -1.86 to -0.30, Rp range: 0.55 to 550.65
Initial guess for DF49 (Year 2021, Dry Period): Rmin=0.0110, d=0.5000, b=0.6000


/Users/estefaniaroldannicolau/Documents/GitHub/Thesis/Paper_1/ANALYSIS/utils/ivc.py:560: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Tree DF49 (Year 2021, Rain Period): PSYx range: -1.19 to -0.24, Rp range: 0.31 to 155.76
Initial guess for DF49 (Year 2021, Rain Period): Rmin=0.0110, d=0.5000, b=0.6000


/Users/estefaniaroldannicolau/Documents/GitHub/Thesis/Paper_1/ANALYSIS/utils/ivc.py:560: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Processing tree: DF21, Year: 2021, Potential Type: matric, SF Type: incomplete
Processing tree: DF27, Year: 2021, Potential Type: matric, SF Type: incomplete
Tree DF27 (Year 2021, Dry Period): PSYx range: -4.66 to -0.64, Rp range: 23.85 to 60934.62
Initial guess for DF27 (Year 2021, Dry Period): Rmin=0.0239, d=0.5000, b=0.6000


...

/Users/estefaniaroldannicolau/Documents/GitHub/Thesis/Paper_1/ANALYSIS/utils/ivc.py:560: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Tree ES42 (Year 2022, Rain Period): PSYx range: -3.53 to -0.02, Rp range: 5.20 to 120741.20
Initial guess for ES42 (Year 2022, Rain Period): Rmin=0.0110, d=0.5000, b=0.6000


/Users/estefaniaroldannicolau/Documents/GitHub/Thesis/Paper_1/ANALYSIS/utils/ivc.py:560: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Processing tree: ES01, Year: 2022, Potential Type: matric, SF Type: incomplete
No data for Dry period for tree ES01 in year 2022. Skipping...
Tree ES01 (Year 2022, Rain Period): PSYx range: -1.73 to -0.24, Rp range: 3220.33 to 34786.40
Initial guess for ES01 (Year 2022, Rain Period): Rmin=0.9000, d=0.5000, b=0.6000


Processing tree: ES51, Year: 2022, Potential Type: matric, SF Type: incomplete

Results for matric potential and incomplete sap flow (Dry and Rain Periods):
    Tree  Year Period       Mean_Rp        Min_Rp         Max_Rp  \
0   DF49  2021    Dry     29.617002      0.552513     550.651637   
1   DF49  2021   Rain     15.947292      0.308851     155.764716   
2   DF27  2021    Dry  17559.444508     23.851802   60934.617552   
3   DF27  2021   Rain   7769.793712     23.816729   19046.655919   
4   ES50  2021    Dry  13667.882075      5.599896  109050.422340   
5   ES50  2021   Rain   4640.392407      7.270541   27204.413695   
6   ES51  2021    Dry  74376.468976  29463.768450  243813.580845   
7   ES51  2021   Rain  47963.662647  29262.454245   99618.471232   
8   DF49  2022    Dry    208.872227      0.264726     492.916879   
9   DF49  2022   Rain    136.917188      0.268908    1300.745489   
10  DF21  2022    Dry  11805.116665    692.432184  109649.810118   
11  DF21  2022   Rain   599

In [6]:
from utils.ivc_modified import calculate_resistance_dry_rain
# Define paths
path_weibull = path_output_graphs + 'IVC_plots/Weibull_fit/'

# Calculate resistance and get figures
results_cdf, plots = calculate_resistance_dry_rain(
    trees_data,
    potential_type='matric',
    sf='incomplete'
)

# Save the plots
for path in [path_weibull, path_latex]:
    for fig, tree_name, year in plots:
        if fig is not None:  # Ensure the figure is valid
            fig.write_image(
                path + f'weibull_fit_matric_{tree_name}_{year}.png',
                scale=8
            )


#results_cdf=calculate_resistance_dry_rain(trees_data, potential_type='matric', sf='incomplete')

Processing tree: DF49, Year: 2021, Potential Type: matric, SF Type: incomplete


AttributeError: 'NoneType' object has no attribute 'get'

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Assuming results_df is the output from calculate_resistance_dry_rain
# Remove duplicates since Dry and Rain periods share the same fit parameters
results_unique = results_cdf.drop_duplicates(subset=['Tree', 'Year', 'Potential_Type'])

# Define the Weibull and HCL functions (same as in your new approach)
def weibull_function(psy, rmin, d, b):
    return rmin * np.exp(((-psy / d) ** b))

def weibull_cdf(psy, d, b):
    return (1 - np.exp(-((-psy / d) ** b))) * 100

# Initialize figures for Rp and HCL plots
fig_all = go.Figure()
fig_2021 = go.Figure()
fig_2022 = go.Figure()

fig_hcl = go.Figure()
fig_hcl_2021 = go.Figure()
fig_hcl_2022 = go.Figure()

# Generate PSYx values for plotting
x = np.linspace(-12, 0.0, 100)

# Loop through unique tree/year/potential_type combinations
for _, row in results_unique.iterrows():
    tree = row['Tree']
    year = str(row['Year'])  # Ensure year is a string for comparison
    rmin = row['Weibull_Rmin']
    d = row['Weibull_d']
    b = row['Weibull_b']
    potential_type = row['Potential_Type'] #matric
    
    # Skip if fit parameters are invalid
    if any(np.isnan([rmin, d, b])):
        continue
    
    # Compute Rp and HCL for the PSYx range
    y = weibull_function(x, rmin, d, b)
    hcl = weibull_cdf(x, d, b)  # Use the CDF for HCL as in the new approach
    
    # Create label for the trace
    label = f"{potential_type}-{tree}-{year}"
    
    # Add to 2021 figures
    if year == '2021':
        fig_2021.add_trace(go.Scatter(x=x, y=y, mode='lines', name=label))
        fig_hcl_2021.add_trace(go.Scatter(x=x, y=hcl, mode='lines', name=label))
    
    # Add to 2022 figures
    elif year == '2022':
        fig_2022.add_trace(go.Scatter(x=x, y=y, mode='lines', name=label))
        fig_hcl_2022.add_trace(go.Scatter(x=x, y=hcl, mode='lines', name=label))
    
    # Add to all-years figures
    fig_all.add_trace(go.Scatter(x=x, y=y, mode='lines', name=label))
    fig_hcl.add_trace(go.Scatter(x=x, y=hcl, mode='lines', name=label))

# Update layouts for Rp plots with y-axis limit
fig_2021.update_layout(
    title='Weibull Fit for 2021',
    xaxis_title='PSYx (MPa)',
    yaxis_title='Rp (MPa s kg⁻¹)',
    yaxis_range=[0, 60000]  # Limit y-axis for Rp plots
)
fig_2021.show()

fig_2022.update_layout(
    title='Weibull Fit for 2022',
    xaxis_title='PSYx (MPa)',
    yaxis_title='Rp (MPa s kg⁻¹)',
    yaxis_range=[0, 60000]  # Limit y-axis for Rp plots
)
fig_2022.show()

fig_all.update_layout(
    title='Weibull Fit for All Years',
    xaxis_title='PSYx (MPa)',
    yaxis_title='Rp (MPa s kg⁻¹)',
    yaxis_range=[0, 60000]  # Limit y-axis for Rp plots
)
fig_all.show()

# Update layouts for HCL plots with y-axis limit
fig_hcl_2021.update_layout(
    title='HCL for 2021',
    xaxis_title='PSYx (MPa)',
    yaxis_title='HCL (%)',
    yaxis_range=[0, 100]  # Already set, kept for clarity
)
fig_hcl_2021.show()

fig_hcl_2022.update_layout(
    title='HCL for 2022',
    xaxis_title='PSYx (MPa)',
    yaxis_title='HCL (%)',
    yaxis_range=[0, 100]  # Already set, kept for clarity
)
fig_hcl_2022.show()

fig_hcl.update_layout(
    title='HCL for All Years',
    xaxis_title='PSYx (MPa)',
    yaxis_title='HCL (%)',
    yaxis_range=[0, 100]  # Already set, kept for clarity
)
fig_hcl.show()

In [ ]:
from utils.ivc_tracheid import calculate_ivc_fits

path_output_graphs = str(path_cwd) + '/Output_graphs/'

results, figs = calculate_ivc_fits(trees_data, path_output=path_output_graphs)




Processing tree: DF49, Year: 2021, Potential Type: matric, SF Type: incomplete
Best R_max for DF49 (Dry): 2625.00 MPa s kg^-1 (Initial R_max: 10500.00, R-squared: -733.42, d: 4.00, b: 6.00)
Best R_max for DF49 (Post-Dry): 2625.00 MPa s kg^-1 (Initial R_max: 10500.00, R-squared: -6242.22, d: 4.00, b: 6.00)
Debug: Adjusted HCL_data(%) for DF49 (Dry) - Min: 0.00, Max: 20.74
Debug: Adjusted HCL_data(%) for DF49 (Post-Dry) - Min: 0.00, Max: 5.65



Processing tree: DF21, Year: 2021, Potential Type: matric, SF Type: incomplete
No SF_W_SM_PSY data for DF21 in year 2021. Skipping...

Processing tree: DF27, Year: 2021, Potential Type: matric, SF Type: incomplete
Best R_max for DF27 (Pre-Dry): 10000.00 MPa s kg^-1 (Initial R_max: 40000.00, R-squared: -2.09, d: 4.00, b: 6.00)
Best R_max for DF27 (Dry): 5000.00 MPa s kg^-1 (Initial R_max: 20000.00, R-squared: 0.35, d: 2.43, b: 1.00)
Best R_max for DF27 (Post-Dry): 12500.00 MPa s kg^-1 (Initial R_max: 20000.00, R-squared: 0.05, d: 3.59, b: 6.00)
Debug: Adjusted HCL_data(%) for DF27 (Pre-Dry) - Min: 18.81, Max: 100.00
Debug: Adjusted HCL_data(%) for DF27 (Dry) - Min: 0.00, Max: 100.00
Debug: Adjusted HCL_data(%) for DF27 (Post-Dry) - Min: 0.00, Max: 100.00



Processing tree: DF03, Year: 2021, Potential Type: matric, SF Type: incomplete
No PSY_cleaned data for DF03 in year 2021. Skipping...

Processing tree: ES48, Year: 2021, Potential Type: matric, SF Type: incomplete
No SF_W_SM_PSY data for ES48 in year 2021. Skipping...

Processing tree: ES50, Year: 2021, Potential Type: matric, SF Type: incomplete
Best R_max for ES50 (Pre-Dry): 12500.00 MPa s kg^-1 (Initial R_max: 50000.00, R-squared: -22.17, d: 4.00, b: 3.55)
Best R_max for ES50 (Dry): 15000.00 MPa s kg^-1 (Initial R_max: 60000.00, R-squared: 0.36, d: 4.00, b: 2.11)
Best R_max for ES50 (Post-Dry): 10000.00 MPa s kg^-1 (Initial R_max: 40000.00, R-squared: -0.96, d: 4.00, b: 6.00)
Debug: Adjusted HCL_data(%) for ES50 (Pre-Dry) - Min: 0.00, Max: 100.00
Debug: Adjusted HCL_data(%) for ES50 (Dry) - Min: 7.77, Max: 100.00
Debug: Adjusted HCL_data(%) for ES50 (Post-Dry) - Min: 0.00, Max: 100.00



Processing tree: ES42, Year: 2021, Potential Type: matric, SF Type: incomplete
No PSY_cleaned data for ES42 in year 2021. Skipping...

Processing tree: ES51, Year: 2021, Potential Type: matric, SF Type: incomplete
Best R_max for ES51 (Dry): 150000.00 MPa s kg^-1 (Initial R_max: 600000.00, R-squared: -0.55, d: 4.00, b: 6.00)
Best R_max for ES51 (Post-Dry): 150000.00 MPa s kg^-1 (Initial R_max: 600000.00, R-squared: -1.98, d: 4.00, b: 6.00)
Debug: Adjusted HCL_data(%) for ES51 (Dry) - Min: 0.00, Max: 100.00
Debug: Adjusted HCL_data(%) for ES51 (Post-Dry) - Min: 13.76, Max: 100.00



Processing tree: DF49, Year: 2022, Potential Type: matric, SF Type: incomplete
Best R_max for DF49 (Pre-Dry): 2625.00 MPa s kg^-1 (Initial R_max: 10500.00, R-squared: -2415.01, d: 4.00, b: 6.00)
Best R_max for DF49 (Dry): 2625.00 MPa s kg^-1 (Initial R_max: 10500.00, R-squared: -472.38, d: 4.00, b: 6.00)
Best R_max for DF49 (Post-Dry): 2625.00 MPa s kg^-1 (Initial R_max: 10500.00, R-squared: -131.13, d: 4.00, b: 6.00)
Debug: Adjusted HCL_data(%) for DF49 (Pre-Dry) - Min: 0.00, Max: 10.70
Debug: Adjusted HCL_data(%) for DF49 (Dry) - Min: 0.00, Max: 18.53
Debug: Adjusted HCL_data(%) for DF49 (Post-Dry) - Min: 0.00, Max: 49.40


...


Processing tree: ES48, Year: 2022, Potential Type: matric, SF Type: incomplete
Best R_max for ES48 (Pre-Dry): 7500.00 MPa s kg^-1 (Initial R_max: 30000.00, R-squared: -4914.38, d: 4.00, b: 6.00)
Best R_max for ES48 (Post-Dry): 750.00 MPa s kg^-1 (Initial R_max: 3000.00, R-squared: -0.64, d: 4.00, b: 6.00)
Debug: Adjusted HCL_data(%) for ES48 (Pre-Dry) - Min: 0.00, Max: 6.39
Debug: Adjusted HCL_data(%) for ES48 (Post-Dry) - Min: 0.00, Max: 100.00



Processing tree: ES50, Year: 2022, Potential Type: matric, SF Type: incomplete
Best R_max for ES50 (Pre-Dry): 12500.00 MPa s kg^-1 (Initial R_max: 50000.00, R-squared: -1498.22, d: 4.00, b: 6.00)
Best R_max for ES50 (Post-Dry): 10000.00 MPa s kg^-1 (Initial R_max: 40000.00, R-squared: -30.74, d: 4.00, b: 6.00)
Debug: Adjusted HCL_data(%) for ES50 (Pre-Dry) - Min: 0.00, Max: 8.08
Debug: Adjusted HCL_data(%) for ES50 (Post-Dry) - Min: 0.00, Max: 90.88



Processing tree: ES42, Year: 2022, Potential Type: matric, SF Type: incomplete
Best R_max for ES42 (Pre-Dry): 75000.00 MPa s kg^-1 (Initial R_max: 300000.00, R-squared: -14.22, d: 4.00, b: 6.00)
Best R_max for ES42 (Post-Dry): 75000.00 MPa s kg^-1 (Initial R_max: 300000.00, R-squared: -230.67, d: 4.00, b: 6.00)
Debug: Adjusted HCL_data(%) for ES42 (Pre-Dry) - Min: 0.00, Max: 100.00
Debug: Adjusted HCL_data(%) for ES42 (Post-Dry) - Min: 0.00, Max: 30.77



Processing tree: ES01, Year: 2022, Potential Type: matric, SF Type: incomplete
Best R_max for ES01 (Pre-Dry): 75000.00 MPa s kg^-1 (Initial R_max: 300000.00, R-squared: -38.57, d: 4.00, b: 6.00)
Debug: Adjusted HCL_data(%) for ES01 (Pre-Dry) - Min: 4.06, Max: 46.25



Processing tree: ES51, Year: 2022, Potential Type: matric, SF Type: incomplete
No PSY_cleaned data for ES51 in year 2022. Skipping...

Results for matric potential and incomplete sap flow:
    Tree    Period  Initial_R_max     R_max  Weibull_d  Weibull_b  \
0   DF49       Dry          10500    2625.0   4.000000   6.000000   
1   DF49  Post-Dry          10500    2625.0   4.000000   6.000000   
2   DF27   Pre-Dry          40000   10000.0   4.000000   6.000000   
3   DF27       Dry          20000    5000.0   2.428440   1.000000   
4   DF27  Post-Dry          20000   12500.0   3.593517   6.000000   
5   ES50   Pre-Dry          50000   12500.0   4.000000   3.550944   
6   ES50       Dry          60000   15000.0   4.000000   2.110970   
7   ES50  Post-Dry          40000   10000.0   4.000000   6.000000   
8   ES51       Dry         600000  150000.0   4.000000   6.000000   
9   ES51  Post-Dry         600000  150000.0   4.000000   6.000000   
10  DF49   Pre-Dry          10500    2625.0   4.000